# 01 · Extract GA4 sample data from BigQuery → local Parquet

Downloads `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*` (92 daily tables) into `data/raw/ga4/`.
Run the cells top to bottom. Re-running is safe: days already downloaded are skipped.

In [ ]:
%pip install -q "google-cloud-bigquery[bqstorage,pandas]>=3.25" "pyarrow>=16" "duckdb>=1.1" "pydata-google-auth>=1.8"

In [ ]:
import sys
from datetime import date
from pathlib import Path

import pydata_google_auth
from google.cloud import bigquery

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "python" / "scripts"))
import ga4_extract  # noqa: E402

GCP_PROJECT = "ambient-future-500915-c1"  # your sandbox project ID

In [ ]:
# Opens a browser window to sign in with your Google account (cached after the first time).
credentials = pydata_google_auth.get_user_credentials(
    ["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project=GCP_PROJECT, credentials=credentials)

In [ ]:
# Test with 3 days first
for d in ga4_extract.daterange(date(2020, 11, 1), date(2020, 11, 3)):
    ga4_extract.extract_day(client, d, force=False)

In [ ]:
# Full download: all 92 days (2020-11-01 → 2021-01-31)
total = sum(
    ga4_extract.extract_day(client, d, force=False)
    for d in ga4_extract.daterange(ga4_extract.DATA_START, ga4_extract.DATA_END)
)
print(f"{total:,} new rows written")

In [ ]:
# Verify with DuckDB
import duckdb

glob = (PROJECT_ROOT / "data" / "raw" / "ga4" / "*" / "events.parquet").as_posix()
src = f"read_parquet('{glob}', hive_partitioning = true)"

duckdb.sql(f"""
    SELECT count(*) AS events, count(DISTINCT user_pseudo_id) AS users,
           min(event_date) AS first_day, max(event_date) AS last_day,
           count(DISTINCT event_date) AS days
    FROM {src}
""").df()

In [ ]:
duckdb.sql(f"SELECT event_name, count(*) AS n FROM {src} GROUP BY 1 ORDER BY n DESC").df()